# CSV Processing for Readability

This notebook processes the csv files produced from the random search so that it includes interesting relevant properties such as unimodality and defectivity.

**Warning:** One should be cautious to check if an architecture is truly minimal. The column "is_minimal" is in reference to the search *thus far*. It is possible for it to actually not be minimal and filling because the search has not uncovered a subarchitecture that is filling yet. One should manually check minimality by computing the dimension of the subarchitectures.

In [14]:
import pandas as pd
import math
import ast
import os
from pathlib import Path
from is_unimodal import is_unimodal


## Helper Functions

In [15]:
# lists of architectures to consider (d0,dh,exponent)

ARCHS = [(d0, dh, r) for d0 in range(1, 25) for dh in range(1, 25) for r in range(1, 25)] 

In [16]:
def expdim(d: list[int], r) -> int | float:
    """
    Calculates the expected dimension of the architecture
    """

    if type(d)==str:
        d = ast.literal_eval(d)
    
    h = len(d) - 1
    d_0 = d[0]
    d_h = d[h]

    # Term 1: d_h + sum_{i=1}^{h} (d_{i-1}-1)*d_i
    sum_term = sum((d[i-1]-1)*d[i] for i in range(1,h+1))
    sum_term += d_h

    # Term 2: d_h * Binomial(r**{h-1} + d_0 - 1, d_0 - 1)
    n = r**(h-1) + d_0 - 1
    k = d_0 - 1
    
    ambient_term = d_h * math.comb(n, k)

    return min(sum_term, ambient_term)

## Processing

In [ ]:
missing = []
empty = []

for entry in ARCHS:
    d0, dh, r = entry

    input_file = f'../data/raw/{d0}_{dh}_r{r}_architectures.csv'

    if not os.path.exists(input_file):
        missing.append(input_file)
        continue

    try:
        df = pd.read_csv(input_file)
    except pd.errors.EmptyDataError:
        empty.append(input_file)
        continue

    if df.empty:  # header only, no rows
        empty.append(input_file)
        continue

    output_file = f'../data/processed/{d0}_{dh}_r{r}_cleaned_architectures.csv'

    df['is_unimodal'] = df['architecture'].apply(is_unimodal)
    df['expected_dimension'] = df['architecture'].apply(lambda x: expdim(x, r))
    df['defect'] = df['expected_dimension'] - df['dimension_computed']
    df = df.sort_values(by=['h', 'is_minimal', 'is_unimodal'], ascending=[True, False, True])

    df.to_csv(output_file, index=False)

print(f"\nSkipped {len(missing)} missing files.")
print(f"Skipped {len(empty)} empty files.")
if empty:
    print("Empty:", *empty, sep="\n  ")

C:\Users\daoke\AppData\Local\Temp\ipykernel_25444\1624224504.py:14: DtypeWarning: Columns (0: shrunk_hidden_layers) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


KeyboardInterrupt: 

# CSV Of Counterexamples

Below, we extra the list of counterexamples to the minimal unimodal conjecture and save it to a separate .csv file.

In [21]:
import pandas as pd

ce = pd.DataFrame()

for entry in ARCHS:
    d0, dh, r = entry

    input_file = f'../data/raw/{d0}_{dh}_r{r}_architectures.csv'

    if input_file in empty:
        continue
    
    if not os.path.exists(input_file):
        missing.append(input_file)
        continue

    df = pd.read_csv(input_file)

    df['is_unimodal'] = df['architecture'].apply(is_unimodal)
    df['expected_dimension'] = df['architecture'].apply(lambda x: expdim(x, r))
    df['defect'] = df['expected_dimension'] - df['dimension_computed'] 
    
    df['d0'] = d0
    df['dh'] = dh
    df['r'] = r

    df = df.sort_values(by=['h','is_minimal','is_unimodal'], ascending=[True,False,True])

    # Filter for counterexamples (fixed the missing closing brackets)
    counterexamples = df[(df['is_minimal'] == True) & 
                         (df['is_full_dimension'] == True) & 
                         (df['is_unimodal'] == False)]
    
    # Save/append to the master CE dataframe
    if not counterexamples.empty:
        ce = pd.concat([ce, counterexamples], ignore_index=True)

ce.to_csv('../data/processed/counterexamples.csv', index=False)
display(ce)

C:\Users\daoke\AppData\Local\Temp\ipykernel_25444\730673924.py:16: DtypeWarning: Columns (0: shrunk_hidden_layers) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,expected_dimension,defect_expected,...,primes,elapsed_seconds,status,covered_by_mfa,is_unimodal,shrunk_hidden_layers,defect,d0,dh,r
0,7.0,2.0,"[2, 3, 4, 5, 4, 6, 4, 1]",110.0,65.0,65.0,True,True,65,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,2,1,2
1,8.0,2.0,"[2, 3, 3, 5, 7, 6, 7, 4, 1]",181.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,2,1,2
2,8.0,2.0,"[2, 3, 3, 6, 7, 6, 7, 3, 1]",183.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,2,1,2
3,8.0,2.0,"[2, 3, 4, 5, 7, 6, 7, 3, 1]",181.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,2,1,2
4,8.0,2.0,"[2, 3, 3, 4, 7, 6, 8, 4, 1]",181.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,2,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,4.0,2.0,"[4, 30, 14, 19, 2]",844.0,330.0,330.0,True,True,330,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,4,2,2
925,5.0,2.0,"[5, 55, 53, 54, 35, 1]",7977.0,4845.0,4845.0,True,True,4845,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,5,1,2
926,5.0,2.0,"[5, 54, 55, 51, 52, 1]",8749.0,4845.0,4845.0,True,True,4845,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,5,1,2
927,5.0,2.0,"[5, 25, 55, 51, 53, 1]",7061.0,4845.0,4845.0,True,True,4845,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,5,1,2


# CSV of Hypersurfaces

In [23]:
import pandas as pd

ce = pd.DataFrame()

for entry in ARCHS:
    d0, dh, r = entry

    input_file = f'../data/raw/{d0}_{dh}_r{r}_architectures.csv'

    if not os.path.exists(input_file):
        missing.append(input_file)
        continue


    if input_file in empty:
        continue

    df = pd.read_csv(input_file)

    df['is_unimodal'] = df['architecture'].apply(is_unimodal)
    df['expected_dimension'] = df['architecture'].apply(lambda x: expdim(x, r))
    df['defect'] = df['expected_dimension'] - df['dimension_computed'] 
    
    df['d0'] = d0
    df['dh'] = dh
    df['r'] = r

    # Filter for hypersurfaces (fixed the missing closing brackets)
    hypersurfaces = df[(df['dimension_computed'] == df['ambient_dimension']-1)]
    
    # Save/append to the master CE dataframe
    if not hypersurfaces.empty:
        ce = pd.concat([ce, hypersurfaces], ignore_index=True)


ce = ce.sort_values(by=['h','dimension_computed','num_parameters'], ascending=[True,True,True])
ce.to_csv('../data/processed/codimension_one.csv', index=False)
display(ce)

C:\Users\daoke\AppData\Local\Temp\ipykernel_25444\4117346933.py:18: DtypeWarning: Columns (0: shrunk_hidden_layers) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,expected_dimension,defect_expected,...,primes,elapsed_seconds,status,covered_by_mfa,is_unimodal,shrunk_hidden_layers,defect,d0,dh,r
0,2.0,2.0,"[2, 1, 1]",3.0,2.0,3.0,False,False,2,NaN,...,NaN,NaN,NaN,NaN,True,NaN,0.0,2,1,2
1471,2.0,1.0,"[2, 1, 2]",4.0,3.0,4.0,False,False,3,NaN,...,NaN,NaN,NaN,NaN,False,NaN,0.0,2,2,1
1105,2.0,4.0,"[2, 2, 1]",6.0,4.0,5.0,False,False,4,NaN,...,NaN,NaN,NaN,NaN,True,NaN,0.0,2,1,4
1661,2.0,2.0,"[3, 2, 1]",8.0,5.0,6.0,False,False,6,NaN,...,NaN,NaN,NaN,NaN,True,NaN,1.0,3,1,2
1414,2.0,6.0,"[2, 3, 1]",9.0,6.0,7.0,False,False,6,NaN,...,NaN,NaN,NaN,NaN,True,NaN,0.0,2,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,10.0,2.0,"[2, 6, 7, 9, 10, 11, 14, 7, 10, 8, 1]",727.0,512.0,513.0,False,False,513,NaN,...,NaN,NaN,NaN,NaN,False,NaN,1.0,2,1,2
492,10.0,2.0,"[2, 4, 8, 4, 9, 7, 14, 8, 16, 14, 1]",747.0,512.0,513.0,False,False,513,NaN,...,NaN,NaN,NaN,NaN,False,NaN,1.0,2,1,2
487,10.0,2.0,"[2, 6, 7, 4, 9, 10, 16, 8, 11, 14, 1]",752.0,512.0,513.0,False,False,513,NaN,...,NaN,NaN,NaN,NaN,False,NaN,1.0,2,1,2
488,10.0,2.0,"[2, 4, 7, 12, 11, 10, 14, 7, 12, 7, 1]",775.0,512.0,513.0,False,False,513,NaN,...,NaN,NaN,NaN,NaN,False,NaN,1.0,2,1,2


# CSV of Minimal Filling Architectures

In [24]:
import pandas as pd

mfa = pd.DataFrame()

for entry in ARCHS:
    d0, dh, r = entry

    input_file = f'../data/raw/{d0}_{dh}_r{r}_architectures.csv'

    if not os.path.exists(input_file):
        missing.append(input_file)
        continue

    if input_file in empty:
        continue
    df = pd.read_csv(input_file)

    df['is_unimodal'] = df['architecture'].apply(is_unimodal)
    df['expected_dimension'] = df['architecture'].apply(lambda x: expdim(x, r))
    df['defect'] = df['expected_dimension'] - df['dimension_computed'] 
    
    df['d0'] = d0
    df['dh'] = dh
    df['r'] = r

    # Filter for MFAs
    mfas = df[(df['is_minimal'] == True)]
    
    # Save/append to the master CE dataframe
    if not mfas.empty:
        mfa = pd.concat([mfa, mfas], ignore_index=True)


mfa = mfa.sort_values(by=['h','exponent','d0', 'dh'], ascending=[True,True,True,True])
mfa.to_csv('../data/processed/MFAs.csv', index=False)
display(mfa)

C:\Users\daoke\AppData\Local\Temp\ipykernel_25444\502662812.py:16: DtypeWarning: Columns (0: shrunk_hidden_layers) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,expected_dimension,defect_expected,...,primes,elapsed_seconds,status,covered_by_mfa,shrunk_hidden_layers,is_unimodal,defect,d0,dh,r
0,2.0,1.0,"[1, 1, 1]",2.0,1.0,1.0,True,True,1,0.0,...,"(10000019, 19511957)",2.643445,filling_below_recorded_mfa,[[2]],NaN,True,0.0,1,1,1
52,2.0,1.0,"[1, 1, 2]",3.0,2.0,2.0,True,True,2,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,1,2,1
122,2.0,1.0,"[1, 1, 3]",4.0,3.0,3.0,True,True,3,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,1,3,1
192,2.0,1.0,"[1, 1, 4]",5.0,4.0,4.0,True,True,4,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,1,4,1
228,2.0,1.0,"[1, 1, 5]",6.0,5.0,5.0,True,True,5,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,1,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,10.0,2.0,"[2, 5, 8, 10, 7, 7, 15, 9, 16, 10, 1]",803.0,513.0,513.0,True,True,513,NaN,...,NaN,NaN,NaN,NaN,NaN,False,0.0,2,1,2
1146,10.0,2.0,"[2, 3, 5, 12, 11, 11, 10, 11, 10, 7, 1]",741.0,513.0,513.0,True,True,513,NaN,...,NaN,NaN,NaN,NaN,NaN,False,0.0,2,1,2
1147,10.0,2.0,"[2, 3, 4, 12, 9, 8, 12, 11, 14, 11, 1]",793.0,513.0,513.0,True,True,513,NaN,...,NaN,NaN,NaN,NaN,NaN,False,0.0,2,1,2
1148,10.0,2.0,"[2, 3, 6, 7, 13, 9, 16, 7, 20, 7, 1]",817.0,513.0,513.0,True,True,513,NaN,...,NaN,NaN,NaN,NaN,NaN,False,0.0,2,1,2


In [25]:
display(mfa[(mfa['h']==8) & (mfa['exponent']==2) & (mfa['dimension_computed']==129) & (mfa['d0']==2) & (mfa['dh']==1) & (mfa['r']==2)])

,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,expected_dimension,defect_expected,...,primes,elapsed_seconds,status,covered_by_mfa,shrunk_hidden_layers,is_unimodal,defect,d0,dh,r
362,8.0,2.0,"[2, 3, 3, 4, 7, 7, 7, 2, 1]",169.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,2,1,2
363,8.0,2.0,"[2, 3, 3, 4, 5, 8, 8, 2, 1]",169.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,2,1,2
364,8.0,2.0,"[2, 3, 3, 4, 6, 8, 6, 3, 1]",168.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,2,1,2
365,8.0,2.0,"[2, 3, 3, 4, 6, 8, 7, 2, 1]",171.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,2,1,2
366,8.0,2.0,"[2, 3, 3, 4, 7, 8, 5, 3, 1]",169.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,2,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,8.0,2.0,"[2, 3, 4, 5, 5, 5, 9, 6, 1]",193.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,2,1,2
486,8.0,2.0,"[2, 3, 3, 6, 6, 6, 7, 7, 1]",203.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,True,0.0,2,1,2
487,8.0,2.0,"[2, 3, 4, 6, 6, 5, 7, 7, 1]",199.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,False,0.0,2,1,2
488,8.0,2.0,"[2, 3, 3, 6, 6, 5, 8, 8, 1]",211.0,129.0,129.0,True,True,129,NaN,...,NaN,NaN,NaN,NaN,NaN,False,0.0,2,1,2


In [26]:
for index,row in mfa[(mfa['h']==8) & (mfa['exponent']==2) & (mfa['dimension_computed']==129) & (mfa['d0']==2) & (mfa['dh']==1) & (mfa['r']==2)].iterrows():
    arch = ast.literal_eval(row["architecture"]) if isinstance(
            row["architecture"], str
        ) else row["architecture"]
    print(max(arch))

7
8
8
8
8
8
8
8
8
8
8
8
8
9
9
9
9
9
9
9
9
9
10
10
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
8
8
8
8
8
8
8
8
8
8
8
8
8
8
9
9
9
9
9
9
9
9
9
9
10
7
7
7
7
7
7
7
7
7
7
8
8
8
8
8
8
8
8
8
8
8
8
8
9
9
9
9
9
9
10
10
10
7
7
7
7
7
8
8
8
8
8
8
8
8
8
8
9
9
9
6
7
7
8
8
8
9
9
9
7
7
8
8


In [27]:
L=8
r=2
d0=2
dh=1

def bound(L, i, r, d0, dh):
                return (1, min(dh * r**(i*d0), math.comb(r**(L-i) + d0 - 1, r**(L-i))))

custom_bounds = {L: [bound(L, i, r, d0, dh) for i in range(L - 1, 0, -1)] for L in range(1, 15)}


In [28]:
custom_bounds[8]

[(1, 3), (1, 5), (1, 9), (1, 17), (1, 33), (1, 16), (1, 4)]